In [1]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [2]:
from fsspec import json
import json

from researchos.paths import SAMPLES_DIR

path_examples = SAMPLES_DIR / 'eval_dataset.json'

with open(path_examples, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [5]:
data

[{'question': 'What are the three coupled dimensions of externalization in LLM agents as described by Zhou et al. (2026)?',
  'reference_answer': 'The three coupled dimensions are memory (externalized state), skills (externalized procedural expertise), and protocols (externalized interaction structure).',
  'source_paper': 'chenyu_zhou_2026.pdf'},
 {'question': "What is the primary function of the 'harness' in an externalized agent architecture?",
  'reference_answer': 'The harness is the engineering layer that coordinates memory, skills, and protocols into governed execution, providing the orchestration logic, constraints, observability, and feedback loops necessary for practical agency.',
  'source_paper': 'chenyu_zhou_2026.pdf'},
 {'question': 'How does externalized memory transform the cognitive task for a Large Language Model?',
  'reference_answer': 'Externalization transforms a difficult internal recall problem (regenerating knowledge from latent weights) into an external recogn

In [4]:
type(data)

list

In [12]:
data[0]['question']

'What are the three coupled dimensions of externalization in LLM agents as described by Zhou et al. (2026)?'

In [15]:
from researchos.domain.interfaces import VectorStore
from researchos.domain.models import Document
from researchos.infrastructure.retrieval.chroma import ChromaVectorStore
from researchos.infrastructure.retrieval.embedder import LocalEmbedder

async def answer_question(question, store: VectorStore, max_results: int = 10) -> list[Document]:
    results = await store.search(query=question, k=max_results)
    for r in results:
        print(f"\nscore: {r.score:.3f}")
        print(f"text: {r.text[:200]}")

    return results


embedder = LocalEmbedder()

store = ChromaVectorStore(
        embedder=embedder,
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
answers = []
for dict_question in data:
    print('-'*10 + f"{dict_question['question']}" + '*'*10)
    print(f"- pdf_ref: {dict_question['source_paper']}")
    answer = await answer_question(dict_question['question'], store=store, max_results=3)
    print(f"- papers in answer: {set([doc.metadata['paper_id'] for doc in answer])}")
    print('\n')

    answers.append(answers)

----------What are the three coupled dimensions of externalization in LLM agents as described by Zhou et al. (2026)?**********
- pdf_ref: chenyu_zhou_2026.pdf

score: 0.786
text: Externalization in LLM Agents: A Unified Review of
Memory, Skills, Protocols and Harness Engineering
Chenyu Zhou1, Huacan Chai1,∗, Wenteng Chen1,∗, Zihan Guo2,3,∗, Rong Shan1,∗, Yuanyi
Song1,∗, Tianyi

score: 0.770
text: dscape onto three capability layers—Weights, Context, and Harness. Fig-
ure 3 complements this view with an architectural overview of the externalized agent, showing the harness
at the center with the

score: 0.763
text: the strongest forms of externalization in agent systems, because it removes entire classes of
reasoning from the critical path. The transformation is analogous to the shift that memory introduces for

- papers in answer: {'chenyu_zhou_2026'}


----------What is the primary function of the 'harness' in an externalized agent architecture?**********
- pdf_ref: chenyu_zhou_2026.pd

In [ ]:
import asyncio
answers = await asyncio.gather(*(answer_question(dict_question['question'], store=store, max_results=3) for dict_question in data))

<coroutine object answer_question at 0x000001DAE1E03780>

In [23]:
[doc.metadata['paper_id'] for doc in answer]

['sam_musker_2024', 'sam_musker_2024', 'sam_musker_2024']